<a href="https://colab.research.google.com/github/kausarfatima2626/recylink1/blob/main/main_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# FastAPI application script Likhein
fastapi_code = """
from fastapi import FastAPI, File, UploadFile, HTTPException
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from PIL import Image
import io
import json

app = FastAPI(title="RecyLink AI - Material Classification API")

# 1. Device Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Material Classes & Mappings
MATERIAL_CLASSES = [
    "pcb", "cable", "battery", "crt", "lcd_display",
    "motor", "magnet_assembly", "mixed_plastics", "other"
]

MATERIAL_TO_CATEGORY_MAP = {
    "pcb": "Information Technology and Telecommunication Equipment (ITE)",
    "cable": "Information Technology and Telecommunication Equipment (ITE)",
    "battery": "Small Electrical and Electronic Equipment",
    "crt": "Consumer Electrical and Electronics and Photovoltaic Panels",
    "lcd_display": "Consumer Electrical and Electronics and Photovoltaic Panels",
    "motor": "Large Electrical and Electronic Equipment",
    "magnet_assembly": "Electrical and Electronic Tools",
    "mixed_plastics": "Small Electrical and Electronic Equipment",
    "other": "Toys, Leisure, and Sports Equipment"
}

# 3. Model Architecture Load Karo
def load_trained_model(weights_path="/content/exported_model/mobilenet_ewaste_classifier.pth"):
    weights = models.MobileNet_V2_Weights.DEFAULT
    model = models.mobilenet_v2(weights=weights)

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2),
        nn.Linear(in_features, 128),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(128, len(MATERIAL_CLASSES))
    )

    if os.path.exists(weights_path):
        model.load_state_dict(torch.load(weights_path, map_location=device))
        print("✅ Model weights successfully loaded for API!")
    else:
        print("⚠️ Model weights not found, running with initial weights for testing.")

    model.to(device)
    model.eval()
    return model

model = load_trained_model()

# Image Preprocessing Pipeline
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

@app.get("/")
def home():
    return {"status": "online", "message": "RecyLink Vision API is running!"}

# 4. Final Endpoint for Person 1 Deliverable
@app.post("/predict")
async def predict_material(file: UploadFile = File(...)):
    if not file.content_type.startswith("image/"):
        raise HTTPException(status_code=400, detail="Uploaded file is not an image.")

    try:
        # Image byte array read karke PIL image me convert karo
        image_bytes = await file.read()
        raw_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

        input_tensor = preprocess(raw_image).unsqueeze(0).to(device)

        with torch.no_grad():
            outputs = model(input_tensor)
            probabilities = F.softmax(outputs, dim=1)
            conf, pred_idx = torch.max(probabilities, 1)

        predicted_class_key = MATERIAL_CLASSES[pred_idx.item()]
        confidence_score = round(float(conf.item()), 4)

        regulatory_cat = MATERIAL_TO_CATEGORY_MAP.get(predicted_class_key, "Unknown")
        display_material = predicted_class_key.upper() if len(predicted_class_key) <= 3 else predicted_class_key.replace("_", " ").title()

        return {
            "material": display_material,
            "regulatory_category": regulatory_cat,
            "confidence": confidence_score
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Inference error: {str(e)}")
"""

with open("/content/app.py", "w") as f:
    f.write(fastapi_code)

print("✅ `app.py` (FastAPI file) Google Colab par save ho gaya hai!")

In [ ]:
!pip install -q fastapi uvicorn requests python-multipart

import uvicorn
import threading
import requests
import time

# Function to run uvicorn server in a separate background thread
def run_api():
    uvicorn.run("app:app", host="127.0.0.1", port=8000, log_level="error")

# Start server in background thread
server_thread = threading.Thread(target=run_api, daemon=True)
server_thread.start()

time.sleep(3) # Server setup delay

# Test Local Endpoint via Request
url = "http://127.0.0.1:8000/predict"
sample_img_path = os.path.join(BASE_DIR, "test", "pcb", os.listdir(os.path.join(BASE_DIR, "test", "pcb"))[0])

with open(sample_img_path, "rb") as img_file:
    files = {"file": ("test.jpg", img_file, "image/jpeg")}
    response = requests.post(url, files=files)

print("🎉 FastAPI Server Direct Test Result:")
print(response.json())